In [1]:
import os
import numpy as np, pandas as pd

needed = ["tft_best_champion.pth", "tft_model.py", "sequences_train.npz",
          "sequences_validation.npz", "sequences_test.npz",
          "target_sales_scaler.joblib"]
for f in needed:
    print(("OK      " if os.path.exists(f) else "MISSING ") + f)

tr = np.load("sequences_train.npz", mmap_mode="r")
df_w = pd.DataFrame({
    "store_nbr": tr["store_nbr"],
    "item_nbr":  tr["item_nbr"],
    "avg_sales": tr["y_raw"].mean(axis=1),
})
series_avg = df_w.groupby(["store_nbr", "item_nbr"])["avg_sales"].mean()

w_series = (series_avg / series_avg.mean()) ** 0.5
w_series = w_series.clip(0.5, 4.0)

print("\nSeries average sales:")
print(series_avg.describe())
print("\nWeights:")
print(w_series.describe())
print("\nSeries at the 4.0 cap:", int((w_series >= 4.0).sum()))

OK      tft_best_champion.pth
OK      tft_model.py
OK      sequences_train.npz
OK      sequences_validation.npz
OK      sequences_test.npz
OK      target_sales_scaler.joblib

Series average sales:
count    400.000000
mean      49.554859
std       58.164318
min        0.829086
25%       16.115767
50%       30.677670
75%       60.443284
max      495.529877
Name: avg_sales, dtype: float64

Weights:
count    400.000000
mean       0.913035
std        0.435425
min        0.500000
25%        0.570271
50%        0.786807
75%        1.104411
max        3.162218
Name: avg_sales, dtype: float64

Series at the 4.0 cap: 0


In [3]:
import torch, numpy as np
from tft_model import TemporalFusionTransformer

ckpt = torch.load("tft_best_champion.pth", map_location="cpu", weights_only=False)
print("checkpoint keys:", list(ckpt.keys()))
print("epoch:", ckpt.get("epoch"), "| best_val_wape:", ckpt.get("best_val_wape"))
print("best_params:", ckpt.get("best_params"))

tr = np.load("sequences_train.npz", mmap_mode="r")
static_cols = ["store_nbr", "item_nbr", "family", "class",
               "perishable", "city", "state", "type", "cluster"]
cards = [len(np.unique(tr[c])) for c in static_cols]

model = TemporalFusionTransformer(
    static_cardinalities=cards, hidden_size=32, embedding_size=16,
    num_heads=2, dropout=0.25, forecast_horizon=7, num_quantiles=1)

try:
    res = model.load_state_dict(ckpt["model_state_dict"])
    print("LOAD RESULT:", res)
except RuntimeError as e:
    msg = str(e)
    print("LOAD FAILED. First 600 chars of error:\n", msg[:600])
else:
    model.eval()
    with torch.no_grad():
        out = model(torch.randn(4, 28, 18), torch.randn(4, 7, 14),
                    torch.zeros(4, 9, dtype=torch.long))
    print("output shape:", tuple(out.shape))
    print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

checkpoint keys: ['model_state_dict', 'best_params', 'best_val_wape', 'epoch']
epoch: 17 | best_val_wape: 29.84
best_params: {'lr': 0.0029621516588303515, 'hidden_size': 64, 'num_heads': 4, 'dropout': 0.25, 'weight_decay': 2.6587543983272695e-05}
LOAD FAILED. First 600 chars of error:
 Error(s) in loading state_dict for TemporalFusionTransformer:
	size mismatch for static_vsn.variable_grns.0.fc1.weight: copying a param with shape torch.Size([64, 16]) from checkpoint, the shape in current model is torch.Size([32, 16]).
	size mismatch for static_vsn.variable_grns.0.fc1.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([32]).
	size mismatch for static_vsn.variable_grns.0.fc2.weight: copying a param with shape torch.Size([64, 64]) from checkpoint, the shape in current model is torch.Size([32, 32]).
	size mismatch for stat
